In [1]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-ma4jxj_3/polara_426a3335244e45b0bcfcfe81cc5dd51b
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-ma4jxj_3/polara_426a3335244e45b0bcfcfe81cc5dd51b
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from typing import Callable, Dict, List, Tuple, Any, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda, split_and_reindex)

In [47]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [48]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history)
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [3]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128

In [ ]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [51]:
histories = train.sort_values([data_description['users'], data_description['timestamp']], ascending=False).groupby(by=data_description['users'])[data_description['items']].apply(list).to_dict()
targets = test.sort_values([data_description['users'], data_description['timestamp']], ascending=False).groupby(by=data_description['users'])[data_description['items']].apply(list).to_dict()

In [52]:
ml_train_dataset = TransfromerTrainDataset(histories=histories)
ml_eval_dataset = TransfromerEvalDataset(histories=histories, targets=targets)

In [ ]:
ml_train_dataloader = DataLoader(
    dataset=ml_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True
)

ml_eval_dataloader = DataLoader(
    dataset=ml_eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False
)

In [ ]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "yambda_interactions.parquet", # из ноута Кирилла (мб кто подкинет ссылку)
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "amzn-books.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [5]:
ml_df = load_ml20m(config["ml-20m"]["interactions_path"], config=config["ml-20m"])
yambda_df = load_yambda(config["yambda"]["interactions_path"], config=config["yambda"])
amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [6]:
ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml-20m"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [7]:
datasets = {
    "ml-20m": {
        "train": ml_train,
        "test": ml_test,
        "desc": ml_data_description,
    },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    "amzn-books": {
        "train": amzn_train,
        "test": amzn_test,
        "desc": amzn_data_description,
    },
}

In [12]:
for ds in datasets.values():
    display(ds['train'].head())

,user_id,item_id,feedback,timestamp
0,0,1,3.5,1112486027
1,0,28,3.5,1112484676
2,0,31,3.5,1112484819
3,0,46,3.5,1112484727
4,0,49,3.5,1112484580


,user_id,item_id,timestamp,feedback
0,0,445451,15929455,1
1,1,412534,18111700,1
2,1,601521,18111700,1
3,1,448958,18111945,1
4,2,595667,106910,1


,user_id,item_id,feedback,timestamp
0,3570759,1769419,5.0,1441260345000
1,3570759,2250799,5.0,1441260365000
2,3570759,1763042,5.0,1523093714024
3,3570759,2699599,1.0,1611623223325
4,3570759,2586413,3.0,1612044209266


In [8]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,ml-20m,18000236,353582,18353818,0.981,0.019,125041,4959,17951
1,yambda,8130564,783082,8913646,0.912,0.088,79910,54555,603402
2,amzn-books,26225396,753938,26979334,0.972,0.028,9330118,427694,3962995
